# Index

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from index.inverted_index import Index
from index.documents import load_dataset, tokenizer


# index = Index().build_index(rcv1, tokenizer)

In [ ]:
# rcv1 = load_dataset("rcv1")

In [ ]:
# i = 0
# x = rcv1.data[i]
# y = rcv1.target[i]
# sid = rcv1.sample_id[i]

# print("sid", sid)
# print("x", x)
# print("y", y)

In [ ]:
# from sklearn.cluster import MiniBatchKMeans

# X = rcv1.data # CSR matrix (804414 x 47236)
# mbk = MiniBatchKMeans(n_clusters=20, batch_size=1000, random_state=0)
# mbk.fit(X)                          # supports sparse input

# # cluster centers are dense (20 x n_features)
# centers = mbk.cluster_centers_
# # top features per cluster
# import numpy as np
# for i, c in enumerate(centers):
#     top_idxs = np.argsort(-c)[:10]
#     print("cluster", i, "top feature indices:", top_idxs)

In [ ]:
msmarco = load_dataset("msmarco-passage/dev/judged")

In [ ]:
print(msmarco)

In [ ]:
import locale, sys
print(sys.flags.utf8_mode)                 # should be 1
print(locale.getpreferredencoding(False))  # should be UTF-8

In [ ]:
msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=10001)

In [ ]:
from scipy.sparse import csr_matrix
import numpy as np

def calculate_tf_idf(index, doc_id):
    tf_idf_vector = {}
    for term in index.get_terms():
        tf = index.get_term_frequency(term, doc_id)
        idf = index.get_inverse_document_frequency(term)
        tf_idf_vector[term] = tf * idf
    return tf_idf_vector

def create_tf_idf_matrix(index):
    tf_idf_matrix = {}
    for doc_id in index.get_doc_ids():
        tf_idf_matrix[doc_id] = calculate_tf_idf(index, doc_id)
    return tf_idf_matrix

def create_sparse_tf_idf_matrix(index, dtype=np.float32):
    terms = list(index.get_terms())
    term_to_col = {t:i for i,t in enumerate(terms)}
    doc_ids = index.get_doc_ids()
    doc_to_row = {did:i for i,did in enumerate(doc_ids)}

    rows, cols, data = [], [], []
    for term, posting in index.index.items():
        idf = index.get_inverse_document_frequency(term)
        col = term_to_col[term]
        for doc_id, tf in posting:
            rows.append(doc_to_row[doc_id])
            cols.append(col)
            data.append(np.log10(1+tf) * idf)

    data = np.array(data, dtype=dtype)
    mat = csr_matrix((data, (rows, cols)), shape=(len(doc_ids), len(terms)), dtype=dtype)
    return mat, doc_ids, terms

In [ ]:
# tf_idf_matrix = create_tf_idf_matrix(msmarco_index)
# print("TF-IDF vector for document 1:", tf_idf_matrix[1])
# print("tf idf values", list(tf_idf_matrix.values().__iter__())[:10])

mat, doc_ids, terms = create_sparse_tf_idf_matrix(msmarco_index)
print("Sparse TF-IDF matrix shape:", mat.shape)
print("Document IDs:", doc_ids[:10])
print("Terms:", terms[:10])

# bytearray(b'\x00\x01\x02')

In [ ]:
import pandas as pd

# df = pd.DataFrame(tf_idf_matrix).T
# print("TF-IDF DataFrame shape:", df.shape)

df = pd.DataFrame.sparse.from_spmatrix(mat, index=doc_ids, columns=terms)
print("DF shape:", df.shape)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

norm_df = normalize(df, norm="l2") #! cosa fa , è fondamentele se non la metti clusteing fa schifo 

kmeans = KMeans(n_clusters=10, random_state=0) #! use jaccard distance instead of euclidean

clusters = kmeans.fit(norm_df)
print("Cluster centers:", kmeans.cluster_centers_)

document_clusters = kmeans.labels_
print("Document clusters:", document_clusters)

In [ ]:
medoids = []

for i in range(kmeans.n_clusters):
    cluster_docs = np.flatnonzero(document_clusters == i)
    cluster_vectors = norm_df[cluster_docs]
    medoid_idx = np.argmin(np.linalg.norm(cluster_vectors - kmeans.cluster_centers_[i], axis=1))
    medoids.append(cluster_docs[medoid_idx])

# medoid_vectors = norm_df[medoids].toarray()

medoid_vectors = norm_df[medoids]
nonzero_cols = medoid_vectors.getnnz(axis=0) > 0
medoid_vectors = medoid_vectors[:, nonzero_cols].toarray()

print("Medoids:", medoids)
print("Medoid vectors shape:", medoid_vectors.shape)
print("Medoid vectors:", medoid_vectors)

# for i in range(10):
#     cluster_docs = [doc_id for doc_id, cluster in enumerate(document_clusters) if cluster == i]
#     new_docs_ordering[i] = cluster_docs
#     print(f"Cluster {i} has {len(cluster_docs)} documents.")

# from itertools import islice

# # medoids contiene row index della matrice (0-based)
# target_rows = set(medoids)
# medoid_docs = {}

# for row_idx, doc in enumerate(islice(msmarco.docs_iter(), len(doc_ids))):
#     if row_idx in target_rows:
#         medoid_docs[row_idx] = doc
#     if len(medoid_docs) == len(target_rows):
#         break

# # esempio: testo del primo medoid
# print(medoid_docs[medoids[0]].text)

# # opzionale: stampa tutti i medoid
# for m in medoids:
#     print(f"\nMedoid row {m}")
#     print(medoid_docs[m].text[:500])

In [ ]:
from sklearn.metrics import pairwise_distances
import numpy as np

def calculate_distance_matrix(norm_df, medoid_vectors, method="euclidean"):
    # medoid_vectors = norm_df[medoids]  # CSR (k, d)

    if method == "euclidean":
        return pairwise_distances(medoid_vectors, metric="euclidean")
    elif method == "cosine":
        return pairwise_distances(medoid_vectors, metric="cosine")
    elif method == "jaccard":
        # Jaccard in sklearn works on boolean/binary features
        return pairwise_distances((medoid_vectors > 0).astype(np.uint8), metric="jaccard")
    else:
        raise ValueError(f"Unknown method: {method}")

In [ ]:
# medoid_vectors = norm_df[medoids]
# print("Medoid vectors shape:", medoid_vectors.shape)
# print("Medoid vectors:", medoid_vectors)
# medoid_vectors.toarray()  
# print("Medoid vectors:", medoid_vectors)

dist_matrix = calculate_distance_matrix(norm_df, medoid_vectors, method="jaccard")
print("Distance matrix shape:", dist_matrix.shape)
print("Distance matrix:\n", dist_matrix)

In [ ]:
import networkx as nx
tsp = nx.approximation.traveling_salesman_problem

tsp_graph = nx.from_numpy_array(dist_matrix)
tsp_path = tsp(tsp_graph, cycle=False)
print("TSP path (medoid indices):", tsp_path)


In [ ]:
def reorder_documents_by_tsp(document_clusters, medoids, tsp_path):
    """
    Reorder documents by cluster order induced by TSP over medoids.
    Inside each cluster, keep documents in their original order (row order).

    Args:
        document_clusters: array-like of shape (n_docs,), cluster label per doc.
        medoids: list of medoid doc indices (not required for ordering here, kept for API compatibility).
        tsp_path: list of cluster ids in the order returned by TSP.

    Returns:
        ordered_doc_ids: list of document row indices in final order.
        cluster_to_docs: dict {cluster_id: [doc_ids in original order]}.
    """
    document_clusters = np.asarray(document_clusters)

    # Build cluster -> docs preserving natural/original order
    cluster_to_docs = {
        c: np.flatnonzero(document_clusters == c).tolist()
        for c in range(len(medoids))
    }

    # Concatenate clusters following TSP order
    ordered_doc_ids = []
    for c in tsp_path:
        ordered_doc_ids.extend(cluster_to_docs.get(c, []))

    return ordered_doc_ids, cluster_to_docs

In [ ]:
ordered_doc_ids, cluster_docs = reorder_documents_by_tsp(document_clusters, medoids, tsp_path)
print("First 30 docs in reordered list:", ordered_doc_ids[:30])

In [ ]:
from itertools import islice

docs = list(islice(msmarco.docs_iter(), len(document_clusters)))

class OrderedDataset:
    def __init__(self, docs, order):
        self.docs = docs
        self.order = order
    def docs_iter(self):
        for i in self.order:
            yield self.docs[i]

ordered_dataset = OrderedDataset(docs, ordered_doc_ids)
reordered_inverted_index = Index()
reordered_inverted_index.build_index(
    ordered_dataset,
    tokenizer,
    max_num_docs=-1, #! -1 since reordered docs are already limited to 10001 in the original index
    d_gap=True
)

stat_r = reordered_inverted_index.get_approx_size()

msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=10001, d_gap=True)

stat_m = msmarco_index.get_approx_size()
print(f"Original index size: {stat_m} bytes")
print(f"Reordered index size: {stat_r} bytes")

ind_1 = reordered_inverted_index.index
ind_2 = msmarco_index.index

print("Msmarco index size:", len(msmarco_index.index))

In [ ]:
# msmarco_index = Index()
# msmarco_index.clear_index()
# msmarco_index.build_index(msmarco, tokenizer, max_num_docs=10000, d_gap=True)
# print("Msmarco index size:", len(msmarco_index.index))
# msmarco_index.remove_tf()
# msmarco_index.compress_index("gamma")
# msmarco_index.save_index("msmarco_index.bin")

In [ ]:
# test_index = Index()
# test_index.clear_index()
# test_index.load_index("msmarco_index.bin")

In [ ]:
# print("Test index loaded. Number of terms:", len(test_index.index))

In [ ]:
def check_correctness(original_index, loaded_index):
    i = 0
    for term, posting in original_index.index.items():
        # if term not in loaded_index.index:
        #     print(f"Term '{term}' missing in loaded index.")
        #     return False
        st = str(i)
        if posting != loaded_index.index[st]:
            print(f"Posting list mismatch for term '{term}'.")
            print(f"Original: {posting}")
            print(f"Loaded: {loaded_index.index[st]}")
            return False
        i += 1
    print("Original and loaded indices match.")
    return True

# test_index.compress_index("gamma")  # Decompress the loaded index for comparison
# check_correctness(msmarco_index, test_index)

In [ ]:
# Test compressions
import copy

def prepare_index(src_index, method):
    idx = copy.deepcopy(src_index)
    idx.remove_tf()
    idx.compress_index(method)
    return idx

# Original index variants
gamma_index = prepare_index(msmarco_index, "gamma")
delta_index = prepare_index(msmarco_index, "delta")
vb_index = prepare_index(msmarco_index, "varbyte")

# Reordered index variants
gamma_index_r = prepare_index(reordered_inverted_index, "gamma")
delta_index_r = prepare_index(reordered_inverted_index, "delta")
vb_index_r = prepare_index(reordered_inverted_index, "varbyte")

print("\n" + "=" * 70)
print(f"{'ORIGINAL (NOT REORDERED) INDEX':^70}")
print("=" * 70)
print("Uncompressed:")
msmarco_index.remove_tf()  # Ensure TF is removed for size calculation
msmarco_index.get_approx_size()

print("\nGamma Compressed:")
gamma_index.get_approx_size()

print("\nDelta Compressed:")
delta_index.get_approx_size()

print("\nVarbyte Compressed:")
vb_index.get_approx_size()

print("\n" + "=" * 70)
print(f"{'REORDERED INDEX (TSP)':^70}")
print("=" * 70)
print("Uncompressed:")
reordered_inverted_index.remove_tf()  # Ensure TF is removed for size calculation
reordered_inverted_index.get_approx_size()

print("\nGamma Compressed:")
gamma_index_r.get_approx_size()

print("\nDelta Compressed:")
delta_index_r.get_approx_size()

print("\nVarbyte Compressed:")
vb_index_r.get_approx_size()

In [ ]:
from index.inverted_index import Index

test = Index()
num = 4
# res = test._Index__varbyte_encode(num)
# print(f"Varbyte encoding of {num}: {res}")
# for value in test._Index__varbyte_decode(res):
#     print(value)

# byte_val = int(res, 2)
# print(f"Decoded value: {byte_val}")

res_2 = test._Index__gamma_encode(num)
print(f"Gamma encoding of {num}: {res_2}")
test._Index__gamma_decode(res_2)

# res_3 = test._Index__delta_encode(num)
# print(f"Delta encoding of {num}: {res_3}")
# test._Index__delta_decode(res_3)

# d_gaps_list = [3,4,7,21,9,12,2,17]
# res_4_list, outliers, min_ = test.pfor_delta_encode(d_gaps_list) 
# print(f"PForDelta encoding of {d_gaps_list}: {res_4_list}")
# print(f"Outliers: {outliers}")
# print(f"Minimum value: {min_}")
# #! how do i distinguish outliers from non-outliers in the decode? assuming that its encoded as contiguous block of bits

# print(test.pfor_delta_decode(res_4_list, outliers, min_))

In [ ]:
#! test encoding and decoding

from index._compression import IndexCompressor

test = IndexCompressor()

numbers = [1, 2, 3, 4, 5,343, 1000, 1024, 2048, 4096, 8192, 16384, 32768, 65536]
for num in numbers:
    res = test.varbyte_encode(num)
    print(f"Varbyte encoding of {num}: {res}")
    for value in test.varbyte_decode(res):
        print(value)

    res_2 = test.gamma_encode(num)
    print(f"Gamma encoding of {num}: {res_2}")
    for value in test.gamma_decode(res_2):
         print(value)
    # test.gamma_decode(res_2)

    res_3 = test.delta_encode(num)
    print(f"Delta encoding of {num}: {res_3}")
    for value in test.delta_decode(res_3):
        print(value)


In [ ]:
def check_correctness(original_index, loaded_index, with_tf=False):
    original_postings = [
        [gap for gap, _tf in posting_list] if not with_tf else posting_list
        for _, posting_list in original_index.index.items()
    ]
    loaded_postings = [
        posting_list
        for _, posting_list in loaded_index.index.items()
    ]

    if len(original_postings) != len(loaded_postings):
        print(f"Length mismatch: {len(original_postings)} vs {len(loaded_postings)}")
        return False

    for i, (expected, actual) in enumerate(zip(original_postings, loaded_postings)):
        if expected != actual:
            print(f"Posting list mismatch at position {i}")
            print(f"Expected: {expected}")
            print(f"Actual:   {actual}")
            return False

    print("Original and loaded indices match.")
    return True

In [ ]:
# compression / loading / saving test
original_index = Index()
original_index.build_index(msmarco, tokenizer, max_num_docs=100, d_gap=True)

msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=100, d_gap=True)
msmarco_index.compress_index("gamma")
msmarco_index.save_index("msmarco_index_gamma.bin")


msmarco_index.clear_index()
msmarco_index.load_index("msmarco_index_gamma.bin")
print(check_correctness(original_index, msmarco_index, with_tf=True))
msmarco_index.compress_index("delta")
msmarco_index.save_index("msmarco_index_delta.bin")


msmarco_index.clear_index()
msmarco_index.load_index("msmarco_index_delta.bin")
print(check_correctness(original_index, msmarco_index, with_tf=True))
msmarco_index.compress_index("varbyte")
msmarco_index.save_index("msmarco_index_varbyte.bin")


msmarco_index.clear_index()
msmarco_index.load_index("msmarco_index_varbyte.bin")
print(check_correctness(original_index, msmarco_index, with_tf=True))